# 🤖 AI Meal Planner — ML Training Pipeline
**COMP6056001 — Artificial Intelligence | Group A**

Inputs (from EDA output):
- `nhanes_clean_eda.csv` — 8,470 rows, 0% missing on features
- `food_groups_merged.csv` — 2,395 food items

What this notebook does:
1. Preprocess & engineer final features
2. Train a `MultiOutputRegressor(RandomForestRegressor)` to predict 5 daily macro targets
3. Evaluate with MAE, RMSE, R² per target
4. Compare against a baseline (mean predictor)
5. Export `trained_model.pkl` + `model_schema.json` + `label_encoder.pkl` to Drive for Group B

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── PATHS — update if yours differ ────────────────────────────────────────
EDA_FOLDER = '/content/drive/MyDrive/AI/EDA_Results'
OUT_FOLDER = '/content/drive/MyDrive/AI/ML_Outputs'
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_FOLDER, exist_ok=True)
print(f'Input  folder: {EDA_FOLDER}')
print(f'Output folder: {OUT_FOLDER}')

In [ ]:
!pip install -q scikit-learn joblib shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, warnings

from sklearn.ensemble          import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput       import MultiOutputRegressor
from sklearn.preprocessing     import LabelEncoder, StandardScaler
from sklearn.model_selection   import train_test_split, cross_val_score, KFold
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline          import Pipeline
from sklearn.dummy             import DummyRegressor
import shap

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.dpi': 120, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})

PURPLE='#534AB7'; TEAL='#1D9E75'; CORAL='#D85A30'; AMBER='#EF9F27'
PAL = [PURPLE, TEAL, CORAL, AMBER, '#378ADD', '#639922']

def save(name):
    path = os.path.join(OUT_FOLDER, f'{name}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  Saved → {path}')
    plt.show()

print('Libraries loaded ✓')

---
## 1. Load & Verify EDA Outputs

In [ ]:
nhanes = pd.read_csv(os.path.join(EDA_FOLDER, 'nhanes_clean_eda.csv'))
food   = pd.read_csv(os.path.join(EDA_FOLDER, 'food_groups_merged.csv'))

print(f'NHANES loaded:    {nhanes.shape[0]:,} rows × {nhanes.shape[1]} cols')
print(f'Food DB loaded:   {food.shape[0]:,} rows × {food.shape[1]} cols')

# Confirm key columns present
required = ['BMXBMI','has_diabetes','has_hypertension','has_malnutrition',
            'DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE']
missing_cols = [c for c in required if c not in nhanes.columns]
if missing_cols:
    print(f'\n⚠ Missing columns: {missing_cols}')
    print('Re-run the EDA notebook first.')
else:
    print('\n✓ All required columns present')

---
## 2. Preprocessing & Feature Engineering

In [ ]:
# ── 2a. Drop rows missing any target (23.5% as found in EDA) ──
TARGETS  = ['DR1TKCAL','DR1TPROT','DR1TCARB','DR1TTFAT','DR1TFIBE']
TARGET_LABELS = ['Calories (kcal)','Protein (g)','Carbs (g)','Fat (g)','Fiber (g)']

df = nhanes.dropna(subset=TARGETS).copy()
print(f'Rows after dropping missing targets: {len(df):,}  (was {len(nhanes):,})')

# ── 2b. Remove physiologically impossible target values ──
# Calories: 400–6000 kcal/day is realistic range
# Extremely low values are likely fasting/incomplete recall days
before = len(df)
df = df[df['DR1TKCAL'].between(400, 6000)]
df = df[df['DR1TPROT'].between(0, 300)]
df = df[df['DR1TCARB'].between(0, 700)]
df = df[df['DR1TTFAT'].between(0, 400)]
df = df[df['DR1TFIBE'].between(0, 100)]
print(f'Rows after removing impossible values: {len(df):,}  (dropped {before-len(df)})')

# ── 2c. BMI must be valid ──
df = df[df['BMXBMI'].between(10, 70)]
print(f'Rows after BMI filter: {len(df):,}')

In [ ]:
# ── 2d. Feature engineering ──
def bmi_category(bmi):
    if bmi < 18.5: return 'underweight'
    elif bmi < 25: return 'normal'
    elif bmi < 30: return 'overweight'
    else:          return 'obese'

df['bmi_category'] = df['BMXBMI'].apply(bmi_category)

# Encode BMI category as ordinal integer (underweight=0 ... obese=3)
le = LabelEncoder()
le.fit(['underweight','normal','overweight','obese'])
df['bmi_cat_encoded'] = le.transform(df['bmi_category'])

# BMI squared — captures non-linear relationship
df['bmi_squared'] = df['BMXBMI'] ** 2

# Disease burden score (0–3, how many conditions)
df['disease_burden'] = (
    df['has_diabetes'].fillna(0) +
    df['has_hypertension'].fillna(0) +
    df['has_malnutrition'].fillna(0)
).astype(int)

# Add age & gender if available (useful predictors)
optional_features = []
if 'RIDAGEYR' in df.columns:
    df['RIDAGEYR'] = df['RIDAGEYR'].fillna(df['RIDAGEYR'].median())
    optional_features.append('RIDAGEYR')
    print('✓ Age (RIDAGEYR) added as feature')
if 'RIAGENDR' in df.columns:
    df['gender_encoded'] = (df['RIAGENDR'] == 1).astype(int)  # 1=Male, 0=Female
    optional_features.append('gender_encoded')
    print('✓ Gender added as feature')

FEATURES = [
    'BMXBMI', 'bmi_cat_encoded', 'bmi_squared',
    'has_diabetes', 'has_hypertension', 'has_malnutrition', 'disease_burden',
] + optional_features

print(f'\nFinal feature set ({len(FEATURES)} features): {FEATURES}')
print(f'Targets ({len(TARGETS)}): {TARGETS}')
print(f'\nTraining dataset: {len(df):,} rows')

In [ ]:
# ── 2e. Target distribution summary ──
print('Target variable summary (after cleaning):')
print(df[TARGETS].describe().round(1).to_string())

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, (col, label) in enumerate(zip(TARGETS, TARGET_LABELS)):
    axes[i].hist(df[col], bins=40, color=PAL[i], alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[i].axvline(df[col].median(), color='black', linestyle='--', linewidth=1.2, alpha=0.7)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('')
plt.suptitle('Target distributions (training data)', fontsize=13, fontweight='bold')
plt.tight_layout()
save('01_target_distributions')

---
## 3. Train / Test Split

In [ ]:
X = df[FEATURES].fillna(0)   # fill any remaining edge-case nulls
y = df[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set:  {X_train.shape[0]:,} rows')
print(f'Test set:      {X_test.shape[0]:,} rows')
print(f'Features:      {X_train.shape[1]}')
print(f'Targets:       {y_train.shape[1]}')

---
## 4. Baseline Model (Mean Predictor)

Always establish a baseline before training. If our model can't beat 'just predict the mean for everyone', it's useless.

In [ ]:
baseline = MultiOutputRegressor(DummyRegressor(strategy='mean'))
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print('Baseline (mean predictor) MAE:')
baseline_maes = {}
for i, (col, label) in enumerate(zip(TARGETS, TARGET_LABELS)):
    mae = mean_absolute_error(y_test[col], y_pred_base[:, i])
    baseline_maes[col] = mae
    print(f'  {label:<20} MAE = {mae:.2f}')

---
## 5. Train Random Forest Model

In [ ]:
%%time
print('Training RandomForestRegressor...')
rf_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
)
rf_model.fit(X_train, y_train)
print('Training complete ✓')

---
## 6. Evaluate — MAE, RMSE, R²

In [ ]:
y_pred_rf = rf_model.predict(X_test)

results = []
print(f'{'Target':<22} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'vs Baseline':>14}')
print('='*65)
for i, (col, label) in enumerate(zip(TARGETS, TARGET_LABELS)):
    mae  = mean_absolute_error(y_test[col], y_pred_rf[:, i])
    rmse = np.sqrt(mean_squared_error(y_test[col], y_pred_rf[:, i]))
    r2   = r2_score(y_test[col], y_pred_rf[:, i])
    impr = (1 - mae / baseline_maes[col]) * 100
    flag = '✓' if impr > 0 else '✗'
    print(f'{label:<22} {mae:>8.2f} {rmse:>8.2f} {r2:>8.3f}   {flag} {impr:+.1f}%')
    results.append({'target': label, 'MAE': round(mae,2), 'RMSE': round(rmse,2),
                    'R2': round(r2,3), 'vs_baseline_pct': round(impr,1)})

results_df = pd.DataFrame(results)
print(f'\nAverage R²: {results_df["R2"].mean():.3f}')

In [ ]:
# ── Evaluation plots ──
fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for i, (col, label) in enumerate(zip(TARGETS, TARGET_LABELS)):
    actual = y_test[col].values
    pred   = y_pred_rf[:, i]

    # Actual vs Predicted scatter
    axes[0, i].scatter(actual, pred, alpha=0.25, s=8, color=PAL[i])
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    axes[0, i].plot(lims, lims, 'k--', linewidth=1, alpha=0.6)
    axes[0, i].set_xlabel('Actual')
    axes[0, i].set_ylabel('Predicted')
    axes[0, i].set_title(f'{label}\nR²={results_df.iloc[i]["R2"]:.3f}', fontsize=9)

    # Residuals
    residuals = pred - actual
    axes[1, i].hist(residuals, bins=40, color=PAL[i], alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[1, i].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[1, i].axvline(residuals.mean(), color=CORAL, linestyle=':', linewidth=1.2,
                        label=f'Mean: {residuals.mean():.1f}')
    axes[1, i].set_title(f'Residuals\nMAE={results_df.iloc[i]["MAE"]:.1f}', fontsize=9)
    axes[1, i].legend(fontsize=7)

plt.suptitle('Random Forest — Actual vs Predicted & Residuals', fontsize=13, fontweight='bold')
plt.tight_layout()
save('02_model_evaluation')

In [ ]:
# ── MAE comparison: RF vs Baseline ──
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TARGETS))
w = 0.35
rf_maes   = [results_df.iloc[i]['MAE'] for i in range(len(TARGETS))]
base_maes = [baseline_maes[c] for c in TARGETS]

ax.bar(x - w/2, base_maes, w, label='Baseline (mean)', color='#D3D1C7', edgecolor='white')
ax.bar(x + w/2, rf_maes,   w, label='Random Forest',   color=PURPLE,    edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(TARGET_LABELS, fontsize=10)
ax.set_ylabel('MAE')
ax.set_title('MAE: Random Forest vs Baseline')
ax.legend()

for i, (b, r) in enumerate(zip(base_maes, rf_maes)):
    impr = (1 - r/b)*100
    ax.text(i + w/2, r + 1, f'{impr:.0f}%↓', ha='center', fontsize=9,
            color=TEAL, fontweight='bold')

plt.tight_layout()
save('03_mae_comparison')

---
## 7. Cross-Validation (5-Fold)

In [ ]:
%%time
print('Running 5-fold cross-validation on Calories target...')
print('(calories only — full CV on all 5 targets takes ~15 min)')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
single_rf = RandomForestRegressor(
    n_estimators=200, max_depth=12,
    min_samples_split=10, min_samples_leaf=5,
    max_features='sqrt', random_state=42, n_jobs=-1
)

cv_scores = cross_val_score(
    single_rf, X, y['DR1TKCAL'],
    cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1
)
cv_maes = -cv_scores

print(f'\nCalories 5-fold CV MAE:')
for fold_i, mae in enumerate(cv_maes, 1):
    print(f'  Fold {fold_i}: {mae:.2f}')
print(f'  Mean: {cv_maes.mean():.2f} ± {cv_maes.std():.2f}')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), cv_maes, color=PURPLE, alpha=0.85, edgecolor='white')
ax.axhline(cv_maes.mean(), color=CORAL, linestyle='--', linewidth=1.5,
            label=f'Mean MAE: {cv_maes.mean():.1f}')
ax.set_xticks(range(1, 6))
ax.set_xticklabels([f'Fold {i}' for i in range(1, 6)])
ax.set_ylabel('MAE (kcal)')
ax.set_title('5-fold CV — Calories MAE per fold')
ax.legend()
plt.tight_layout()
save('04_cross_validation')

---
## 8. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
feature_labels = {
    'BMXBMI':           'BMI',
    'bmi_cat_encoded':  'BMI category',
    'bmi_squared':      'BMI²',
    'has_diabetes':     'Diabetes',
    'has_hypertension': 'Hypertension',
    'has_malnutrition': 'Malnutrition',
    'disease_burden':   'Disease burden',
    'RIDAGEYR':         'Age',
    'gender_encoded':   'Gender',
}
feat_display = [feature_labels.get(f, f) for f in FEATURES]

for i, (col, label) in enumerate(zip(TARGETS, TARGET_LABELS)):
    importances = rf_model.estimators_[i].feature_importances_
    sorted_idx  = np.argsort(importances)[::-1]
    axes[i].bar(range(len(importances)),
                importances[sorted_idx],
                color=PAL[i], alpha=0.85, edgecolor='white')
    axes[i].set_xticks(range(len(importances)))
    axes[i].set_xticklabels(
        [feat_display[j] for j in sorted_idx],
        rotation=45, ha='right', fontsize=8
    )
    axes[i].set_title(label, fontsize=9)
    axes[i].set_ylabel('Importance' if i == 0 else '')

plt.suptitle('Feature importances per target', fontsize=13, fontweight='bold')
plt.tight_layout()
save('05_feature_importance')

---
## 9. Prediction by Disease Group

This validates that the model is actually learning different patterns per disease — critical for the AoL report.

In [ ]:
# Show what the model predicts for representative users
test_users = pd.DataFrame([
    # bmi, bmi_cat, bmi_sq, diab, hyper, mal, burden  (+age, gender if present)
    {'label': 'Normal BMI, no condition',    'BMXBMI':22.0, 'bmi_cat_encoded':1, 'bmi_squared':22**2, 'has_diabetes':0,'has_hypertension':0,'has_malnutrition':0,'disease_burden':0},
    {'label': 'Overweight, no condition',    'BMXBMI':27.5, 'bmi_cat_encoded':2, 'bmi_squared':27.5**2,'has_diabetes':0,'has_hypertension':0,'has_malnutrition':0,'disease_burden':0},
    {'label': 'Obese, diabetes',             'BMXBMI':34.0, 'bmi_cat_encoded':3, 'bmi_squared':34**2, 'has_diabetes':1,'has_hypertension':0,'has_malnutrition':0,'disease_burden':1},
    {'label': 'Obese, hypertension',         'BMXBMI':34.0, 'bmi_cat_encoded':3, 'bmi_squared':34**2, 'has_diabetes':0,'has_hypertension':1,'has_malnutrition':0,'disease_burden':1},
    {'label': 'Obese, diabetes+hypertension','BMXBMI':36.0, 'bmi_cat_encoded':3, 'bmi_squared':36**2, 'has_diabetes':1,'has_hypertension':1,'has_malnutrition':0,'disease_burden':2},
    {'label': 'Underweight, malnutrition',   'BMXBMI':16.5, 'bmi_cat_encoded':0, 'bmi_squared':16.5**2,'has_diabetes':0,'has_hypertension':0,'has_malnutrition':1,'disease_burden':1},
])

# Add optional features with neutral values
if 'RIDAGEYR' in FEATURES:
    test_users['RIDAGEYR'] = 40
if 'gender_encoded' in FEATURES:
    test_users['gender_encoded'] = 1

X_test_users = test_users[FEATURES]
preds = rf_model.predict(X_test_users)

print(f'{'Profile':<32} {'Calories':>10} {'Protein':>10} {'Carbs':>10} {'Fat':>10} {'Fiber':>10}')
print('='*84)
for row_i, (_, row) in enumerate(test_users.iterrows()):
    p = preds[row_i]
    print(f'{row["label"]:<32} {p[0]:>10.0f} {p[1]:>10.1f} {p[2]:>10.1f} {p[3]:>10.1f} {p[4]:>10.1f}')

# Visualise
pred_df = pd.DataFrame(preds, columns=TARGET_LABELS)
pred_df.insert(0, 'Profile', test_users['label'].values)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, label in enumerate(TARGET_LABELS):
    bars = axes[i].barh(pred_df['Profile'].str[:25], pred_df[label],
                         color=PAL[i], alpha=0.85, edgecolor='white')
    axes[i].set_title(label, fontsize=9)
    axes[i].invert_yaxis()
    for bar, val in zip(bars, pred_df[label]):
        axes[i].text(val + 1, bar.get_y() + bar.get_height()/2,
                     f'{val:.0f}', va='center', fontsize=7)

plt.suptitle('Predicted daily targets by user profile', fontsize=13, fontweight='bold')
plt.tight_layout()
save('06_predictions_by_profile')

---
## 10. Export Model Artifacts for Group B

In [ ]:
# ── 10a. Save trained model ──
model_path = os.path.join(OUT_FOLDER, 'trained_model.pkl')
joblib.dump(rf_model, model_path)
print(f'Saved trained_model.pkl  ({os.path.getsize(model_path)/1e6:.1f} MB)')

# ── 10b. Save label encoder (needed to encode bmi_category in app.py) ──
le_path = os.path.join(OUT_FOLDER, 'label_encoder.pkl')
joblib.dump(le, le_path)
print(f'Saved label_encoder.pkl')

# ── 10c. Save schema (Group B reads this to know exact input format) ──
schema = {
    'features':      FEATURES,
    'targets':       TARGETS,
    'target_labels': TARGET_LABELS,
    'bmi_categories': list(le.classes_),
    'disease_options': ['none', 'diabetes', 'hypertension', 'malnutrition'],
    'input_example': {
        'bmi': 27.4,
        'disease': 'diabetes',
        'ingredients': ['chicken', 'rice', 'spinach']
    },
    'output_example': {
        'DR1TKCAL': 1850, 'DR1TPROT': 85.0,
        'DR1TCARB': 220.0, 'DR1TTFAT': 62.0, 'DR1TFIBE': 28.0
    },
    'model_performance': results_df.to_dict(orient='records')
}
schema_path = os.path.join(OUT_FOLDER, 'model_schema.json')
with open(schema_path, 'w') as f:
    json.dump(schema, f, indent=2)
print(f'Saved model_schema.json')

# ── 10d. Save evaluation results CSV ──
results_df.to_csv(os.path.join(OUT_FOLDER, 'model_evaluation.csv'), index=False)
print(f'Saved model_evaluation.csv')

# ── 10e. Save ml-ready dataset ──
ml_df = df[FEATURES + TARGETS + ['bmi_category']].copy()
ml_df.to_csv(os.path.join(OUT_FOLDER, 'ml_dataset_final.csv'), index=False)
print(f'Saved ml_dataset_final.csv  ({len(ml_df):,} rows)')

In [ ]:
# ── 10f. Quick sanity check — load model and predict one user ──
loaded_model  = joblib.load(model_path)
loaded_le     = joblib.load(le_path)
loaded_schema = json.load(open(schema_path))

def predict_macros(bmi, disease):
    """Simulate what Group B's app.py will call."""
    cat = bmi_category(bmi)
    disease_map = {
        'none':         [0, 0, 0],
        'diabetes':     [1, 0, 0],
        'hypertension': [0, 1, 0],
        'malnutrition': [0, 0, 1],
    }
    d, h, m   = disease_map.get(disease, [0, 0, 0])
    burden    = d + h + m
    cat_enc   = loaded_le.transform([cat])[0]

    row = {'BMXBMI': bmi, 'bmi_cat_encoded': cat_enc,
           'bmi_squared': bmi**2, 'has_diabetes': d,
           'has_hypertension': h, 'has_malnutrition': m,
           'disease_burden': burden}
    if 'RIDAGEYR'      in loaded_schema['features']: row['RIDAGEYR']      = 40
    if 'gender_encoded' in loaded_schema['features']: row['gender_encoded'] = 1

    X_in = pd.DataFrame([row])[loaded_schema['features']]
    pred = loaded_model.predict(X_in)[0]
    return dict(zip(loaded_schema['targets'], [round(float(v), 1) for v in pred]))

print('Sanity check — predict_macros(bmi=27.4, disease="diabetes"):')
print(predict_macros(27.4, 'diabetes'))

print('\npredict_macros(bmi=16.5, disease="malnutrition"):')
print(predict_macros(16.5, 'malnutrition'))

print('\n✓ Model loads and predicts correctly — ready to hand off to Group B')

In [ ]:
# ── Final file listing ──
print(f'\n=== FILES IN {OUT_FOLDER} ===')
for fname in sorted(os.listdir(OUT_FOLDER)):
    size_kb = os.path.getsize(os.path.join(OUT_FOLDER, fname)) / 1024
    print(f'  {fname:<40} {size_kb:>8.0f} KB')

print('\n=== HANDOFF CHECKLIST FOR GROUP B ===')
checklist = [
    ('trained_model.pkl',    'ML model — load with joblib.load()'),
    ('label_encoder.pkl',    'Encodes bmi_category string → integer'),
    ('model_schema.json',    'Feature names, target names, input/output format'),
    ('model_evaluation.csv', 'MAE/RMSE/R² — include in AoL report'),
    ('ml_dataset_final.csv', 'Full training dataset — for verification'),
]
for fname, note in checklist:
    exists = '✓' if os.path.exists(os.path.join(OUT_FOLDER, fname)) else '✗ MISSING'
    print(f'  [{exists}] {fname:<30} {note}')